# P0_FINAL_SOILNET_VICREG_MU27_LI_V4_BESTREG

**Scientific question:** What is the final leakage-controlled performance of SoilNet with LI and verified VICReg μ=27 initialization?  
**Configuration:** `config/experiments/P0_final_soilnet_v4_bestreg.yaml` (protocol revision v4-bestreg)  
**Dataset manifest:** `data/manifests/final_clean_manifest_v1.csv` (1,927 unique images)  
**Split:** 1,407 train / 289 validation / 231 sealed test  
**Checkpoint/initialization:** ImageNet → VICReg μ=27, SHA256 locked before load  
**Expected outputs:** `validation_best_regression.pth` (primary), `epoch_60_final.pth` (endpoint), history, best-checkpoint validation predictions/metrics, metadata, and both SHA256 values in the new external run directory.

Run cells in order. This notebook constructs train and validation loaders only. It never opens the sealed test partition.


## 1. Imports

Import the tested `soilnet` environment and reusable repository modules. No package installation or environment mutation occurs.


In [1]:
from pathlib import Path
import csv, json, os, sys
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import torch
from torch import nn

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / "pyproject.toml").is_file():
    REPO = REPO.parent
if not (REPO / "pyproject.toml").is_file():
    raise RuntimeError("Open this notebook from inside user-home://SoilNet")
sys.path.insert(0, str(REPO / "src"))

from soilnet.models import ExperimentSoilNet, load_verified_soilnet_ssl
from soilnet.training import (
    build_optimizer, build_train_loader, build_val_loader,
    run_one_batch_preflight, train_experiment,
)
from soilnet.utils import load_experiment_context, print_environment


## 2. Repository paths

Resolve the repository and ignored machine-local roots. Binary outputs remain outside Git under `run_artifact_root`.


In [2]:
CONFIG_PATH = REPO / "config/experiments/P0_final_soilnet_v4_bestreg.yaml"
print({"repository": str(REPO), "config": str(CONFIG_PATH)})


{'repository': 'user-home://SoilNet', 'config': 'repo://config/experiments/P0_final_soilnet_v4_bestreg.yaml'}


## 3. Environment information

Print Python, PyTorch, torchvision, timm, CUDA build, GPU visibility, seed, and locked hashes. Python 3.11 is the tested compatible environment; it is not claimed to be the historical Python 3.10 environment.


In [3]:
context = load_experiment_context(CONFIG_PATH)
environment = print_environment(context)


python: 3.11.15
torch: 2.6.0+cu118
torchvision: 0.21.0+cu118
torchaudio: 2.6.0+cu118
timm: 1.0.29
cuda_available: True
cuda_device_count: 1
gpu: NVIDIA GeForce RTX 3050
torch_cuda_runtime: 11.8
seed: 20260905
config_sha256: ad39b3fc8217b297e79c9752cb7d2eaed19872478625f1ef9982a771b2f68e5a
manifest_sha256: 8ff45054d4b8e3df9758d0c112dc16719572b2267906fb3a5ed5b3262a6732bd
split_sha256: 8927b8223b8c4c234d264ad9ea62ac2df6161124a79787a2e71eeb5cd23eae2f


## 4. Verify locked artifacts

Context construction has already recomputed the manifest, split, and μ27 checkpoint hashes. Stop immediately if any locked scientific input differs from P0-v3.


In [4]:
assert context.manifest_sha256 == "8ff45054d4b8e3df9758d0c112dc16719572b2267906fb3a5ed5b3262a6732bd"
assert context.split_sha256 == "8927b8223b8c4c234d264ad9ea62ac2df6161124a79787a2e71eeb5cd23eae2f"
assert context.config["ssl_checkpoint"]["sha256"] == "42599ac025b8d8c8c5b8cca36624665f155e0ff588cd90de2fd47b08cd2d60d8"
assert context.config["epochs"] == 60 and context.config["learning_rate"] == 1e-4
assert context.config["primary_checkpoint"] == "validation_best_regression.pth"
assert context.config["checkpoint_selection"] == "minimum_mean_validation_RMSE"
print("LOCKED_ARTIFACTS: PASS")


LOCKED_ARTIFACTS: PASS


## 5. GPU check

Full P0 training is CUDA-only. If CUDA is unavailable, stop here and diagnose WSL/driver access separately; this notebook never changes the environment.


In [5]:
print("torch.__version__:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.cuda.device_count():", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU_BLOCKED: stop before preflight/training and fix GPU access separately")


torch.__version__: 2.6.0+cu118
torch.version.cuda: 11.8
torch.cuda.is_available(): True
torch.cuda.device_count(): 1
GPU: NVIDIA GeForce RTX 3050


## 6. Dataset and split summary

Read only the locked split manifest to confirm counts. The test count shown is locked metadata; no test row is selected or loaded.


In [6]:
with context.split_path.open(newline="", encoding="utf-8") as handle:
    split_rows = list(csv.DictReader(handle))
split_counts = {name: sum(row["split"] == name for row in split_rows) for name in ("train", "validation", "test")}
assert split_counts == {"train": 1407, "validation": 289, "test": 231}
print({"train": 1407, "validation": 289, "sealed_test_metadata_only": 231})
del split_rows


{'train': 1407, 'validation': 289, 'sealed_test_metadata_only': 231}


## 7. Build train loader

Construct only the 1,407-row training loader with batch size 32 and the locked deterministic transform.


In [7]:
train_loader = build_train_loader(context)
assert len(train_loader.dataset) == 1407
print({"train_samples": len(train_loader.dataset), "batch_size": train_loader.batch_size})


{'train_samples': 1407, 'batch_size': 32}


## 8. Build validation loader

Construct only the 289-row deterministic validation loader. No test-loader helper is imported.


In [8]:
validation_loader = build_val_loader(context)
assert len(validation_loader.dataset) == 289
print({"validation_samples": len(validation_loader.dataset), "test_loader": "NOT_CREATED"})


{'validation_samples': 289, 'test_loader': 'NOT_CREATED'}


## 9. Build SoilNet

Instantiate the canonical dual-head SoilNet structure on CPU. The full verified SSL state is loaded in the next cell.


In [9]:
inspection_model = ExperimentSoilNet(
    num_classes=context.config["num_classes"], use_li_signal=True, backbone_pretrained=False
)
print({"model": inspection_model.__class__.__name__, "regression_outputs": 2, "classes": 10, "LI": True})


{'model': 'ExperimentSoilNet', 'regression_outputs': 2, 'classes': 10, 'LI': True}


## 10. Verify and load μ27 checkpoint

Recompute SHA256, normalize only the audited historical aliases, then require a strict 577/577 canonical load with no residual incompatibility.


In [10]:
ssl = context.config["ssl_checkpoint"]
ssl_path = context.checkpoint_root / ssl["relative_path"]
checkpoint_report = load_verified_soilnet_ssl(inspection_model, ssl_path, ssl["sha256"])
assert checkpoint_report["matched_keys"] == checkpoint_report["canonical_keys"] == 577
assert checkpoint_report["missing_after_normalization"] == []
assert checkpoint_report["unexpected_after_normalization"] == []
assert checkpoint_report["shape_mismatches"] == []
print(checkpoint_report)


{'checkpoint_sha256': '42599ac025b8d8c8c5b8cca36624665f155e0ff588cd90de2fd47b08cd2d60d8', 'matched_keys': 577, 'raw_missing_keys': [], 'raw_unexpected_alias_keys': ['mobilevit_full.head.fc.bias', 'mobilevit_full.head.fc.weight', 'mobilevit_full.stages.0.0.conv1_1x1.bn.bias', 'mobilevit_full.stages.0.0.conv1_1x1.bn.num_batches_tracked', 'mobilevit_full.stages.0.0.conv1_1x1.bn.running_mean', 'mobilevit_full.stages.0.0.conv1_1x1.bn.running_var', 'mobilevit_full.stages.0.0.conv1_1x1.bn.weight', 'mobilevit_full.stages.0.0.conv1_1x1.conv.weight', 'mobilevit_full.stages.0.0.conv2_kxk.bn.bias', 'mobilevit_full.stages.0.0.conv2_kxk.bn.num_batches_tracked', 'mobilevit_full.stages.0.0.conv2_kxk.bn.running_mean', 'mobilevit_full.stages.0.0.conv2_kxk.bn.running_var', 'mobilevit_full.stages.0.0.conv2_kxk.bn.weight', 'mobilevit_full.stages.0.0.conv2_kxk.conv.weight', 'mobilevit_full.stages.0.0.conv3_1x1.bn.bias', 'mobilevit_full.stages.0.0.conv3_1x1.bn.num_batches_tracked', 'mobilevit_full.stages.0.0

## 11. Define losses and optimizer

Historical code uses MSE regression plus CrossEntropy classification with unit weights. Adam uses learning rate 1e-4 and weight decay 0.


In [11]:
regression_criterion = nn.MSELoss()
classification_criterion = nn.CrossEntropyLoss()
inspection_optimizer = build_optimizer(inspection_model, context.config)
print({"total_loss": "regression_loss + classification_loss", "optimizer": inspection_optimizer})


{'total_loss': 'regression_loss + classification_loss', 'optimizer': Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0.0
)}


## 12. One-batch preflight

This optional check builds a separate temporary model, runs one train forward/backward and one validation forward, and performs **no optimizer step**. It cannot alter the final model initialized by the training function.


In [12]:
RUN_PREFLIGHT = True
if RUN_PREFLIGHT:
    if not torch.cuda.is_available():
        raise RuntimeError("GPU_BLOCKED: stop and fix GPU access before the P0 preflight")
    preflight = run_one_batch_preflight(
        context, device="cuda", train_loader=train_loader, val_loader=validation_loader
    )
    assert preflight["optimizer_step_performed"] is False
    assert preflight["temporary_model"] is True
    assert all(preflight["loss_components_finite"].values())
    print({"PREFLIGHT": "PASS", **preflight})
else:
    print("PREFLIGHT_NOT_RUN")


{'PREFLIGHT': 'PASS', 'status': 'PASS', 'device': 'cuda', 'train_batches': 1, 'validation_batches': 1, 'backward_batches': 1, 'train_samples': 1407, 'validation_samples': 289, 'train_image_shape': [32, 3, 224, 224], 'train_li_shape': [32, 1], 'train_regression_target_shape': [32, 2], 'train_class_target_shape': [32], 'output_shapes': [[32, 2], [32, 10]], 'load_report': {'checkpoint_sha256': '42599ac025b8d8c8c5b8cca36624665f155e0ff588cd90de2fd47b08cd2d60d8', 'matched_keys': 577, 'raw_missing_keys': [], 'raw_unexpected_alias_keys': ['mobilevit_full.head.fc.bias', 'mobilevit_full.head.fc.weight', 'mobilevit_full.stages.0.0.conv1_1x1.bn.bias', 'mobilevit_full.stages.0.0.conv1_1x1.bn.num_batches_tracked', 'mobilevit_full.stages.0.0.conv1_1x1.bn.running_mean', 'mobilevit_full.stages.0.0.conv1_1x1.bn.running_var', 'mobilevit_full.stages.0.0.conv1_1x1.bn.weight', 'mobilevit_full.stages.0.0.conv1_1x1.conv.weight', 'mobilevit_full.stages.0.0.conv2_kxk.bn.bias', 'mobilevit_full.stages.0.0.conv2_k

## 13. Training switch

Leave training disabled until hashes, GPU, 577/577 loading, preflight, and the test firewall all pass. Resume is accepted only from this exact v4 experiment when every identity hash matches; P0-v3 can never be resumed here.


In [13]:
RUN_TRAINING = True
RESUME_IF_AVAILABLE = True
print({"RUN_TRAINING": RUN_TRAINING, "RESUME_IF_AVAILABLE": RESUME_IF_AVAILABLE})


{'RUN_TRAINING': True, 'RESUME_IF_AVAILABLE': True}


## 14. Full training

This is the only 60-epoch cell; validation checkpoint selection never early-stops training. It builds a fresh P0 model from the locked μ27 initialization, never uses the preflight model, and refuses CPU fallback or automatic hyperparameter changes.


In [14]:
run_metadata = None
if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("GPU_BLOCKED: full P0 training requires CUDA; CPU fallback is prohibited")
    run_metadata = train_experiment(context, resume_if_available=RESUME_IF_AVAILABLE)
else:
    print("TRAINING_NOT_RUN: manually set RUN_TRAINING=True only after the full preflight passes")


BEST REGRESSION UPDATED | epoch=1 | mean_RMSE=21.4461 | SM0=21.4323 | SM20=21.4598
{"epoch": 1, "train_total_loss": 2.2456001693552192, "train_regression_loss": 0.07105923596430909, "train_classification_loss": 2.174540942365473, "validation_total_loss": 2.1673428058624267, "SM0_RMSE": 21.432289260277376, "SM0_MAE": 16.572079675007856, "SM20_RMSE": 21.45984090913585, "SM20_MAE": 16.459570643811077, "classification_accuracy": 0.2491349480968858, "Macro-F1": 0.20980204888633658, "regression_metric_scale": "original_0_to_100_percentage_points"}
BEST REGRESSION UPDATED | epoch=2 | mean_RMSE=19.8249 | SM0=19.8548 | SM20=19.7950
{"epoch": 2, "train_total_loss": 1.9527459794824773, "train_regression_loss": 0.046762275975197554, "train_classification_loss": 1.905983710830862, "validation_total_loss": 1.985464906692505, "SM0_RMSE": 19.854804206364115, "SM0_MAE": 14.85598614842834, "SM20_RMSE": 19.795000963880682, "SM20_MAE": 14.884524774386396, "classification_accuracy": 0.2837370242214533, "Ma

## 15. Validation evaluation

After epoch 60, training reloads `validation_best_regression.pth` and writes validation predictions and metrics on the original 0–100 percentage-point regression scale. This cell only reads those saved development artifacts; it does not access test data.


In [15]:
validation_metrics_path = context.run_dir / "validation_metrics.json"
validation_metrics = json.loads(validation_metrics_path.read_text(encoding="utf-8")) if validation_metrics_path.is_file() else None
print({"validation_metrics": validation_metrics, "regression_scale": "original_0_to_100_percentage_points", "test_evaluated": False})


{'validation_metrics': {'checkpoint_selection': 'minimum_mean_validation_RMSE', 'classification': {'accuracy': 0.4809688581314879, 'macro_f1': 0.4771764421990395, 'macro_precision': 0.4856271777003484, 'macro_recall': 0.4838881827361803}, 'generated_from_checkpoint': 'release-artifact://P0_FINAL_SOILNET_VICREG_MU27_LI_V4_BESTREG/validation_best_regression.pth', 'generated_from_checkpoint_epoch': 45, 'generated_from_checkpoint_sha256': 'eba009dfd45ec21174a8e40b16148e0455e902286c7db55933487221da761379', 'n': 289, 'regression': {'SM_0': {'mae': 10.299095516798818, 'me': 1.127402097708626, 'r2': 0.7624771522980361, 'rmse': 14.420743601668894}, 'SM_20': {'mae': 10.472837215063894, 'me': 0.8537708883879507, 'r2': 0.754491151518814, 'rmse': 14.636912724403645}}, 'regression_scale': 'original_0_to_100_percentage_points'}, 'regression_scale': 'original_0_to_100_percentage_points', 'test_evaluated': False}


## 16. Save-artifact check

The training engine retains the validation-best primary checkpoint and the epoch-60 endpoint artifact. The rolling resume file is overwritten each epoch and removed after successful epoch 60.


In [16]:
expected_artifacts = [
    "validation_best_regression.pth", "epoch_60_final.pth", "training_history.csv", "validation_predictions.csv",
    "validation_metrics.json", "run_metadata.json", "checkpoint_sha256.txt",
]
print({name: (context.run_dir / name).is_file() for name in expected_artifacts})
print({"external_run_directory": str(context.run_dir), "repository_checkpoint": False})


{'validation_best_regression.pth': True, 'epoch_60_final.pth': True, 'training_history.csv': True, 'validation_predictions.csv': True, 'validation_metrics.json': True, 'run_metadata.json': True, 'checkpoint_sha256.txt': True}
{'external_run_directory': 'release-artifact://P0_FINAL_SOILNET_VICREG_MU27_LI_V4_BESTREG', 'repository_checkpoint': False}


## 17. Experiment Summary

Summarize only saved train/validation evidence. The prospective test remains sealed.


In [17]:
metadata_path = context.run_dir / "run_metadata.json"
summary = json.loads(metadata_path.read_text(encoding="utf-8")) if metadata_path.is_file() else {}
print({
    "experiment_id": "P0_FINAL_SOILNET_VICREG_MU27_LI_V4_BESTREG",
    "training_completed": summary.get("training_completed", False),
    "primary_checkpoint_path": summary.get("primary_checkpoint_path"),
    "primary_checkpoint_sha256": summary.get("primary_checkpoint_sha256"),
    "epoch_60_checkpoint_path": summary.get("epoch_60_checkpoint_path"),
    "epoch_60_checkpoint_sha256": summary.get("epoch_60_checkpoint_sha256"),
    "best_epoch": summary.get("best_epoch"),
    "final_training_metrics": summary.get("final_training_metrics"),
    "validation_metrics": summary.get("validation_metrics"),
    "test_evaluated": "NO",
})


{'experiment_id': 'P0_FINAL_SOILNET_VICREG_MU27_LI_V4_BESTREG', 'training_completed': True, 'primary_checkpoint_path': 'release-artifact://P0_FINAL_SOILNET_VICREG_MU27_LI_V4_BESTREG/validation_best_regression.pth', 'primary_checkpoint_sha256': 'eba009dfd45ec21174a8e40b16148e0455e902286c7db55933487221da761379', 'epoch_60_checkpoint_path': 'release-artifact://P0_FINAL_SOILNET_VICREG_MU27_LI_V4_BESTREG/epoch_60_final.pth', 'epoch_60_checkpoint_sha256': '7aa1036cf4644c31ff0008e651a2af05d6112f8f112bc44830e742358ca96652', 'best_epoch': 45, 'final_training_metrics': {'classification': {'accuracy': 0.9637526652452025, 'macro_f1': 0.9662769355748171, 'macro_precision': 0.9673750444388063, 'macro_recall': 0.9663155998208286}, 'n': 1407, 'regression': {'SM_0': {'mae': 4.389724292417071, 'me': 0.2033564303687217, 'r2': 0.9587192325680829, 'rmse': 5.773497132724114}, 'SM_20': {'mae': 4.95383613484077, 'me': -0.10304923877702508, 'r2': 0.9470559598964642, 'rmse': 6.54917238455951}}, 'regression_scal